# 02 · Modélisation (axe 4)
Six modèles comparés en validation croisée stratifiée à 5 plis sur le **train** : baseline, régression logistique, arbre libre, arbre réglé, Random Forest, HistGradientBoosting. Toute la préparation est dans un `Pipeline`. Le test n'est pas touché ici.

In [1]:
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             roc_auc_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

sys.path.insert(0, os.path.abspath("../src"))
from features import CAT_COLS, NUM_COLS, TARGET, add_features, clean, load_raw
from costs import confusion, metrics, roi, total_cost

warnings.filterwarnings("ignore")
RS = 42
ROOT = os.path.abspath("..")
FIG = os.path.join(ROOT, "reports", "figures")
os.makedirs(FIG, exist_ok=True)

# ---------------- hypothèses business (axes 6-7), toutes testées en sensibilité -------------
LGD, MARGIN = 0.60, 0.12          # perte si défaut = 60 % du montant ; marge nette perdue si refus = 12 %
N_ANNUAL = 32_000                 # nombre de demandes traitées par an (hypothèse = taille du jeu)
COUT_PROJET = 150_000             # 1re année : développement 80 k + intégration/validation 30 k + exploitation 40 k
GAIN_REALISE = 0.70               # décote : seule une part du gain mesuré hors échantillon est réalisée en production
GUARDRAIL_PRECISION = 0.50        # au plus 1 client sain refusé pour 1 défaut évité

NAVY, TEAL, ORANGE, GREY, RED = "#1F3A5F", "#2A9D8F", "#E76F51", "#8D99AE", "#C0392B"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 10, "axes.titleweight": "bold", "figure.dpi": 150})
R = {}                            # tous les résultats
t0 = time.time()

def save(fig, name):
    fig.savefig(os.path.join(FIG, name), bbox_inches="tight"); plt.close(fig)

In [2]:
# ------------------------------------------------------------------ 1. données
raw = load_raw(os.path.join(ROOT, "data", "credit_risk_dataset.csv"))
R["raw"] = {"n": len(raw), "cols": raw.shape[1],
            "missing": raw.isna().sum()[lambda s: s > 0].to_dict(),
            "age_max": float(raw.person_age.max()), "emp_max": float(raw.person_emp_length.max()),
            "income_max": float(raw.person_income.max())}
df, qc = clean(raw)
R["clean"] = qc
df = add_features(df)
R["default_rate"] = float(df[TARGET].mean())
R["n_clean"] = len(df)
R["rate_by"] = {
    "grade": df.groupby("loan_grade")[TARGET].mean().round(3).to_dict(),
    "prior_default": df.groupby("cb_person_default_on_file")[TARGET].mean().round(3).to_dict(),
    "home": df.groupby("person_home_ownership")[TARGET].mean().round(3).to_dict(),
    "intent": df.groupby("loan_intent")[TARGET].mean().round(3).to_dict(),
}
df["lpi_bin"] = pd.qcut(df["loan_percent_income"], 8, duplicates="drop")
R["rate_by"]["lpi_bin_first_last"] = [float(df.groupby("lpi_bin", observed=True)[TARGET].mean().iloc[0]),
                                      float(df.groupby("lpi_bin", observed=True)[TARGET].mean().iloc[-1])]
R["missing_rate_by_default"] = {
    "int_rate_missing": df.assign(m=df.loan_int_rate.isna()).groupby("m")[TARGET].mean().round(3).to_dict()}

# ------------------------------------------------------------------ 2. split (AVANT toute préparation)
X = df[NUM_COLS + CAT_COLS]
y = df[TARGET]
idx_tr, idx_te = train_test_split(df.index, test_size=0.2, stratify=y, random_state=RS)
X_tr, X_te, y_tr, y_te = X.loc[idx_tr], X.loc[idx_te], y.loc[idx_tr], y.loc[idx_te]
amt_tr, amt_te = df.loc[idx_tr, "loan_amnt"].values, df.loc[idx_te, "loan_amnt"].values
R["split"] = {"train": len(X_tr), "test": len(X_te), "default_train": float(y_tr.mean()),
              "default_test": float(y_te.mean())}


def make_prep(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                          ("sc", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])


def make_pipe(clf, num_cols=NUM_COLS, cat_cols=CAT_COLS):
    return Pipeline([("prep", make_prep(num_cols, cat_cols)), ("clf", clf)])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)

## Comparaison des modèles et sélection
Règle fixée à l'avance : meilleur PR-AUC en CV ; si égalité à un écart-type près, modèle le plus simple.

In [3]:
# ------------------------------------------------------------------ 3. modèles (axe 4)
specs = {
    "Baseline (probabilité a priori)": (DummyClassifier(strategy="prior"), {}, 0),
    "Régression logistique": (LogisticRegression(max_iter=2000, class_weight="balanced"),
                              {"clf__C": [0.01, 0.1, 1, 10]}, 1),
    "Arbre sans max_depth": (DecisionTreeClassifier(random_state=RS), {}, 2),
    "Arbre (max_depth réglé)": (DecisionTreeClassifier(random_state=RS, min_samples_leaf=20),
                                {"clf__max_depth": [3, 5, 7, 10]}, 2),
    "Random Forest": (RandomForestClassifier(n_estimators=150, random_state=RS, n_jobs=-1,
                                             min_samples_leaf=3),
                      {"clf__max_depth": [6, 10, 16]}, 3),
    "Gradient Boosting (HGB)": (HistGradientBoostingClassifier(max_iter=250, random_state=RS,
                                                              early_stopping=True),
                                {"clf__max_depth": [3, 5], "clf__learning_rate": [0.05, 0.1]}, 4),
}
scoring = {"ap": "average_precision", "auc": "roc_auc"}
table, fitted, complexity = [], {}, {}
for name, (clf, grid, cx) in specs.items():
    pipe = make_pipe(clf)
    gs = GridSearchCV(pipe, grid if grid else {"clf": [clf]}, scoring=scoring, refit="ap", cv=cv,
                      return_train_score=True, n_jobs=1)
    gs.fit(X_tr, y_tr)
    i = gs.best_index_
    cr = gs.cv_results_
    table.append({"modele": name, "params": {k.replace("clf__", ""): v for k, v in gs.best_params_.items()
                                             if k != "clf"},
                  "ap_train": float(cr["mean_train_ap"][i]), "ap_cv": float(cr["mean_test_ap"][i]),
                  "ap_cv_std": float(cr["std_test_ap"][i]), "auc_cv": float(cr["mean_test_auc"][i]),
                  "auc_cv_std": float(cr["std_test_auc"][i])})
    fitted[name] = gs.best_estimator_
    complexity[name] = cx
    print(f"{name:34s} AP cv {table[-1]['ap_cv']:.3f} ± {table[-1]['ap_cv_std']:.3f} "
          f"(train {table[-1]['ap_train']:.3f})  [{time.time()-t0:.0f}s]", flush=True)
R["models"] = table

# règle de sélection a priori : meilleur PR-AUC en CV ; parmi les modèles à moins d'un écart-type
# du meilleur, on retient le plus simple (règle de l'écart-type)
best_row = max(table, key=lambda r: r["ap_cv"])
ok = [r for r in table if r["ap_cv"] >= best_row["ap_cv"] - best_row["ap_cv_std"]]
chosen = min(ok, key=lambda r: (complexity[r["modele"]], -r["ap_cv"]))["modele"]
R["chosen_model"] = chosen
R["best_by_ap"] = best_row["modele"]
model = fitted[chosen]
print("Modèle retenu :", chosen, flush=True)

Baseline (probabilité a priori)    AP cv 0.219 ± 0.000 (train 0.219)  [1s]


Régression logistique              AP cv 0.708 ± 0.016 (train 0.709)  [5s]


Arbre sans max_depth               AP cv 0.616 ± 0.012 (train 1.000)  [6s]


Arbre (max_depth réglé)            AP cv 0.863 ± 0.004 (train 0.885)  [10s]


Random Forest                      AP cv 0.879 ± 0.005 (train 0.972)  [53s]


Gradient Boosting (HGB)            AP cv 0.899 ± 0.003 (train 0.947)  [83s]


Modèle retenu : Gradient Boosting (HGB)


In [4]:
# fig 1 : schéma du pipeline
fig, ax = plt.subplots(figsize=(9.2, 2.3)); ax.axis("off"); ax.set_xlim(0, 100); ax.set_ylim(0, 24)
boxes = [("Données\nbrutes\n32 580 lignes", NAVY), ("Nettoyage\ndoublons,\nvaleurs\nimpossibles", NAVY),
         ("Split 80/20\nstratifié", ORANGE), ("Pipeline\nsklearn\nimputation,\néchelle,\none-hot", TEAL),
         ("6 modèles\n+ CV 5 plis", TEAL), ("Seuil par\nle coût\n(OOF du\ntrain)", ORANGE),
         ("Test\n(1 seule\nfois)\n+ ROI", NAVY)]
w, gap = 12.4, 2.0
for i, (txt, c) in enumerate(boxes):
    x = 1 + i * (w + gap)
    ax.add_patch(FancyBboxPatch((x, 5), w, 14, boxstyle="round,pad=0.3", fc=c, ec="none", alpha=.92))
    ax.text(x + w / 2, 12, txt, ha="center", va="center", color="white", fontsize=7, fontweight="bold")
    if i < len(boxes) - 1:
        ax.annotate("", xy=(x + w + gap - .2, 12), xytext=(x + w + .3, 12),
                    arrowprops=dict(arrowstyle="-|>", color=GREY, lw=1.4))
ax.text(1 + 2 * (w + gap) + w / 2, 1.2, "le test est mis de côté ici", ha="center", fontsize=7, color=ORANGE)
save(fig, "fig1_pipeline.png")

In [5]:
# fig 4 : comparaison des modèles
fig, axs = plt.subplots(1, 2, figsize=(9, 3.4), gridspec_kw={"width_ratios": [1.3, 1]})
names = [r["modele"] for r in table]; short = ["Baseline", "Rég.\nlogistique", "Arbre\nlibre", "Arbre\nréglé", "Random\nForest", "HGB"]
xs = np.arange(len(names)); wbar = 0.38
axs[0].bar(xs - wbar / 2, [r["ap_train"] for r in table], wbar, label="train", color=GREY)
axs[0].bar(xs + wbar / 2, [r["ap_cv"] for r in table], wbar, yerr=[r["ap_cv_std"] for r in table],
           label="validation croisée (±1 σ)", color=TEAL, capsize=3)
axs[0].set_xticks(xs); axs[0].set_xticklabels(short, fontsize=7); axs[0].set_ylabel("PR-AUC")
axs[0].set_title("L'arbre libre sur-apprend ; RF et HGB généralisent"); axs[0].legend(fontsize=7, frameon=False)
ci = names.index(chosen)
axs[0].annotate("retenu", (ci + wbar / 2, table[ci]["ap_cv"] + table[ci]["ap_cv_std"] + .02), ha="center", color=ORANGE, fontweight="bold")
axs[1].barh(range(len(names)), [r["auc_cv"] for r in table], xerr=[r["auc_cv_std"] for r in table], color=NAVY, capsize=3)
axs[1].set_yticks(range(len(names))); axs[1].set_yticklabels([s.replace("\n", " ") for s in short], fontsize=7)
axs[1].set_xlim(0.4, 1); axs[1].set_xlabel("ROC-AUC (CV)"); axs[1].invert_yaxis(); axs[1].set_title("Même classement en ROC-AUC")
fig.tight_layout(); save(fig, "fig4_models.png")

## Sauvegarde du modèle retenu (utilisé par le notebook 03)

In [6]:
import joblib
os.makedirs(os.path.join(ROOT, "reports"), exist_ok=True)
joblib.dump({"model": model, "chosen": chosen, "table": table}, os.path.join(ROOT, "reports", "model.joblib"))
print("modèle retenu :", chosen)

modèle retenu : Gradient Boosting (HGB)
